# ATLAS — PDF to JSON Converter
**Arquiteto de Transformação e Limpeza de Assets Científicos**

Este notebook converte PDFs científicos em JSONs estruturados, limpos e parseáveis.

## Fluxo
1. Instalar dependências
2. Escanear best-sources e verificar idempotência
3. Converter PDFs → JSON
4. (Opcional) Expandir para outras pastas
5. Gerar relatório de conversão

---
⚠️ **Este notebook é idempotente**: arquivos já convertidos são automaticamente pulados.


## Célula 1 — Instalar dependências
Execute uma vez. Pode pular se já instalado.

In [ ]:
import subprocess
import sys

packages = ['pdfplumber', 'pymupdf', 'pathlib']
for pkg in packages:
    try:
        __import__(pkg if pkg != 'pymupdf' else 'fitz')
        print(f'✅ {pkg} já instalado')
    except ImportError:
        print(f'📦 Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
        print(f'✅ {pkg} instalado com sucesso')

print('\n🚀 Dependências prontas!')

## Célula 2 — Configuração de caminhos
Ajuste `REPO_ROOT` se necessário.

In [ ]:
from pathlib import Path
import os

# Detectar automaticamente a raiz do repositório
# O notebook está em: .agents/skills/pdf-to-json-converter/scripts/
NOTEBOOK_DIR = Path(os.getcwd())

# Subir 4 níveis para chegar à raiz do repo
REPO_ROOT = NOTEBOOK_DIR.parents[3] if 'skills' in str(NOTEBOOK_DIR) else NOTEBOOK_DIR

# Fallback: ajuste manual se necessário
# REPO_ROOT = Path(r'c:\Users\evers\Desktop\corporative-rag-research-picoc')

# Caminhos principais
ARTICLES_ROOT = REPO_ROOT / 'picoc-method' / 'material' / 'articles'
OUTPUT_ROOT   = REPO_ROOT / 'src' / 'scripts_outputs' / 'articles-outputs'

FOLDERS = {
    'best-sources':        ARTICLES_ROOT / 'best-sources',
    'all-sources-filtered': ARTICLES_ROOT / 'all-sources-filtered',
    'alt-sources':         ARTICLES_ROOT / 'alt-sources',
}

# Criar diretórios de output se não existirem
for folder_name in FOLDERS:
    (OUTPUT_ROOT / folder_name).mkdir(parents=True, exist_ok=True)

print(f'📁 Raiz do repositório: {REPO_ROOT}')
print(f'📂 Artigos: {ARTICLES_ROOT}')
print(f'📤 Output: {OUTPUT_ROOT}')
print()

# Verificar existência das pastas
for name, path in FOLDERS.items():
    exists = path.exists()
    pdfs = list(path.glob('*.pdf')) if exists else []
    print(f'  {"✅" if exists else "❌"} {name}: {len(pdfs)} PDFs encontrados')

## Célula 3 — Funções de extração e estruturação

In [ ]:
import json
import re
from datetime import datetime, timezone
from typing import Optional

# ──────────────────────────────────────────────
# Extração de texto via pdfplumber (primário)
# ──────────────────────────────────────────────
def extract_with_pdfplumber(pdf_path: Path) -> tuple[list[str], int, list[str]]:
    """Extrai texto página a página com pdfplumber. Retorna (pages_text, n_pages, warnings)."""
    import pdfplumber
    warnings = []
    pages_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            n_pages = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    pages_text.append(text)
                else:
                    warnings.append(f'Página {i+1}: texto não extraível (pode ser imagem)')
        return pages_text, n_pages, warnings
    except Exception as e:
        return [], 0, [f'pdfplumber falhou: {str(e)}']


# ──────────────────────────────────────────────
# Extração de texto via pymupdf (fallback)
# ──────────────────────────────────────────────
def extract_with_pymupdf(pdf_path: Path) -> tuple[list[str], int, list[str]]:
    """Extrai texto com pymupdf. Retorna (pages_text, n_pages, warnings)."""
    import fitz  # pymupdf
    warnings = []
    pages_text = []
    try:
        doc = fitz.open(str(pdf_path))
        n_pages = len(doc)
        for i, page in enumerate(doc):
            text = page.get_text()
            if text.strip():
                pages_text.append(text)
            else:
                warnings.append(f'Página {i+1}: vazia ou baseada em imagem')
        doc.close()
        return pages_text, n_pages, warnings
    except Exception as e:
        return [], 0, [f'pymupdf falhou: {str(e)}']


# ──────────────────────────────────────────────
# Heurísticas de parsing estrutural
# ──────────────────────────────────────────────
SECTION_PATTERNS = [
    r'^(\d+\.?\s+[A-Z][^\n]{3,60})$',           # "1. Introduction"
    r'^([A-Z][A-Z\s]{4,50})$',                   # "INTRODUCTION"
    r'^((?:Abstract|Introduction|Conclusion|References|Related Work)[:\s])', # palavras-chave
]

def parse_sections(full_text: str) -> list[dict]:
    """Identifica seções heuristicamente. Nunca inventa conteúdo."""
    lines = full_text.split('\n')
    sections = []
    current_heading = 'Preâmbulo'
    current_content = []
    current_level = 0

    for line in lines:
        stripped = line.strip()
        is_heading = False
        level = 1
        for pat in SECTION_PATTERNS:
            if re.match(pat, stripped):
                is_heading = True
                break

        if is_heading and len(stripped) > 3:
            if current_content:
                sections.append({
                    'heading': current_heading,
                    'level': current_level,
                    'content': ' '.join(current_content).strip()
                })
            current_heading = stripped
            current_content = []
            current_level = level
        else:
            if stripped:
                current_content.append(stripped)

    if current_content:
        sections.append({
            'heading': current_heading,
            'level': current_level,
            'content': ' '.join(current_content).strip()
        })

    return sections


def extract_abstract(full_text: str) -> Optional[str]:
    """Extrai abstract heuristicamente. Retorna None se não encontrado."""
    patterns = [
        r'Abstract[:\s—]+(.{100,2000}?)(?=\n\n|\n[A-Z1-9])',
        r'ABSTRACT[:\s—]+(.{100,2000}?)(?=\n\n|\n[A-Z1-9])',
    ]
    for pat in patterns:
        match = re.search(pat, full_text, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).replace('\n', ' ').strip()
    return None


def extract_title(full_text: str, filename: str) -> Optional[str]:
    """Tenta extrair o título das primeiras linhas. Nunca usa o nome do arquivo como fallback."""
    lines = [l.strip() for l in full_text.split('\n')[:20] if l.strip()]
    for line in lines:
        if 10 < len(line) < 200 and not re.match(r'^\d', line):
            return line
    return None


def extract_year(full_text: str) -> Optional[int]:
    """Extrai ano de publicação das primeiras 500 chars."""
    match = re.search(r'\b(20[12]\d)\b', full_text[:500])
    return int(match.group(1)) if match else None


def extract_references(full_text: str) -> list[dict]:
    """Extrai referências brutas sem inventar dados."""
    refs = []
    ref_section_match = re.search(
        r'(?:References|Bibliography|REFERENCES)[\s\n]+(.*?)$',
        full_text, re.DOTALL | re.IGNORECASE
    )
    if not ref_section_match:
        return refs

    ref_text = ref_section_match.group(1)
    raw_refs = re.split(r'\n\[\d+\]|\n\d+\.\s', ref_text)

    for ref in raw_refs:
        ref = ref.strip()
        if len(ref) > 20:
            refs.append({'raw_text': ref, 'parsed': {'authors': [], 'title': None, 'year': None, 'venue': None}})

    return refs[:100]  # Limitar a 100 referências


print('✅ Funções de extração carregadas com sucesso.')

## Célula 4 — Motor de conversão principal

In [ ]:
def convert_pdf_to_json(pdf_path: Path, output_folder: Path, folder_name: str) -> dict:
    """
    Converte um único PDF em JSON estruturado.
    Retorna um dict com status e dados do arquivo gerado.
    NUNCA inventa dados — campos não encontrados recebem null.
    """
    output_path = output_folder / (pdf_path.stem + '.json')

    # Verificar idempotência
    if output_path.exists():
        return {'status': 'skipped_idempotent', 'file': pdf_path.name, 'output': str(output_path)}

    # Tentar extração com pdfplumber primeiro
    pages_text, n_pages, warnings = extract_with_pdfplumber(pdf_path)
    extraction_tool = 'pdfplumber'

    # Fallback para pymupdf se pdfplumber falhou
    if not pages_text:
        warnings.append('pdfplumber retornou vazio — usando pymupdf como fallback')
        pages_text, n_pages, pymupdf_warnings = extract_with_pymupdf(pdf_path)
        warnings.extend(pymupdf_warnings)
        extraction_tool = 'pymupdf'

    # Se ainda não há texto, registrar erro mas NÃO inventar conteúdo
    if not pages_text:
        warnings.append('ERRO: Nenhum texto extraível. Arquivo pode ser scaneado/baseado em imagens.')

    full_text = '\n\n'.join(pages_text)

    # Estruturar JSON — campos null quando não encontrados
    document = {
        'metadata': {
            'source_file': pdf_path.name,
            'source_folder': folder_name,
            'conversion_timestamp': datetime.now(timezone.utc).isoformat(),
            'converter_version': '1.0',
            'extraction_tool': extraction_tool,
            'pages_total': n_pages,
            'extraction_warnings': warnings
        },
        'title': extract_title(full_text, pdf_path.stem) if full_text else None,
        'authors': [],  # Extração de autores é complexa — campo deixado para enriquecimento futuro
        'year': extract_year(full_text) if full_text else None,
        'venue': None,  # Não inferível com segurança sem metadados externos
        'doi': None,    # Não inferível com segurança
        'abstract': extract_abstract(full_text) if full_text else None,
        'keywords': [],
        'sections': parse_sections(full_text) if full_text else [],
        'references': extract_references(full_text) if full_text else [],
        'figures_detected': 0,  # Requer análise de imagem — fora do escopo atual
        'tables_detected': 0,
        'full_text_raw': full_text  # Texto bruto completo para consumo por agentes
    }

    # Salvar JSON
    output_path.write_text(
        json.dumps(document, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )

    status = 'converted' if full_text else 'converted_with_errors'
    return {'status': status, 'file': pdf_path.name, 'output': str(output_path), 'warnings': warnings}


print('✅ Motor de conversão pronto.')

## Célula 5 — Converter best-sources (sempre executada)

In [ ]:
folder_name = 'best-sources'
source_folder = FOLDERS[folder_name]
output_folder = OUTPUT_ROOT / folder_name

pdfs = sorted(source_folder.glob('*.pdf'))
print(f'📂 {folder_name}: {len(pdfs)} PDFs encontrados')
print('─' * 60)

results = []
for pdf in pdfs:
    result = convert_pdf_to_json(pdf, output_folder, folder_name)
    results.append(result)
    icon = {'converted': '✅', 'skipped_idempotent': '⏩', 'converted_with_errors': '⚠️'}.get(result['status'], '❓')
    print(f"{icon} {result['file']}")
    if result.get('warnings'):
        for w in result['warnings']:
            print(f"   ↳ ⚠️  {w}")

print('─' * 60)
converted = sum(1 for r in results if r['status'] == 'converted')
skipped   = sum(1 for r in results if r['status'] == 'skipped_idempotent')
errors    = sum(1 for r in results if r['status'] == 'converted_with_errors')
print(f'\n📊 Resumo best-sources:')
print(f'   ✅ Convertidos: {converted}')
print(f'   ⏩ Pulados (já existiam): {skipped}')
print(f'   ⚠️  Com avisos: {errors}')
print(f'   📁 Output em: {output_folder}')

## Célula 6 — Escolher pastas adicionais (opcional)

In [ ]:
print('=' * 60)
print('Conversão de best-sources concluída.')
print('\nDeseja converter outras pastas?')
print('  [1] all-sources-filtered/ (28 PDFs)')
print('  [2] alt-sources/ (10 PDFs)')
print('  [3] Ambas')
print('  [4] Não, encerrar')
print('='* 60)

choice = input('\nDigite sua escolha [1/2/3/4]: ').strip()

folders_to_process = []
if choice == '1':
    folders_to_process = ['all-sources-filtered']
elif choice == '2':
    folders_to_process = ['alt-sources']
elif choice == '3':
    folders_to_process = ['all-sources-filtered', 'alt-sources']
else:
    folders_to_process = []
    print('\n⏹️  Encerrando. Nenhuma pasta adicional será processada.')

print(f'\nPastas selecionadas: {folders_to_process if folders_to_process else "nenhuma"}')

## Célula 7 — Converter pastas adicionais selecionadas

In [ ]:
all_results = results.copy()

for folder_name in folders_to_process:
    source_folder = FOLDERS[folder_name]
    output_folder = OUTPUT_ROOT / folder_name

    pdfs = sorted(source_folder.glob('*.pdf'))
    print(f'\n📂 {folder_name}: {len(pdfs)} PDFs')
    print('─' * 60)

    folder_results = []
    for pdf in pdfs:
        result = convert_pdf_to_json(pdf, output_folder, folder_name)
        folder_results.append(result)
        all_results.append(result)
        icon = {'converted': '✅', 'skipped_idempotent': '⏩', 'converted_with_errors': '⚠️'}.get(result['status'], '❓')
        print(f"{icon} {result['file']}")

    converted = sum(1 for r in folder_results if r['status'] == 'converted')
    skipped   = sum(1 for r in folder_results if r['status'] == 'skipped_idempotent')
    print(f'   → ✅ {converted} convertidos | ⏩ {skipped} pulados')

print('\n✅ Todas as pastas selecionadas processadas!')

## Célula 8 — Gerar relatório de conversão

In [ ]:
# Verificar critério de parada: há PDFs restantes não convertidos?
remaining_pdfs = []
for folder_name, folder_path in FOLDERS.items():
    if folder_path.exists():
        output_folder = OUTPUT_ROOT / folder_name
        pdfs = list(folder_path.glob('*.pdf'))
        for pdf in pdfs:
            json_out = output_folder / (pdf.stem + '.json')
            if not json_out.exists():
                remaining_pdfs.append({'folder': folder_name, 'file': pdf.name})

# Gerar relatório
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
report = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'folders_processed': ['best-sources'] + folders_to_process,
    'total_pdfs_found': len(all_results),
    'total_converted': sum(1 for r in all_results if r['status'] == 'converted'),
    'total_skipped_idempotent': sum(1 for r in all_results if r['status'] == 'skipped_idempotent'),
    'total_errors': sum(1 for r in all_results if r['status'] == 'converted_with_errors'),
    'remaining_unconverted': len(remaining_pdfs),
    'remaining_files': remaining_pdfs,
    'output_path': str(OUTPUT_ROOT),
    'details': all_results
}

report_path = OUTPUT_ROOT / f'conversion_report_{timestamp}.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print('=' * 60)
print('📋 RELATÓRIO FINAL DE CONVERSÃO')
print('=' * 60)
print(f"  📁 Pastas processadas : {report['folders_processed']}")
print(f"  📄 PDFs encontrados   : {report['total_pdfs_found']}")
print(f"  ✅ Convertidos        : {report['total_converted']}")
print(f"  ⏩ Pulados            : {report['total_skipped_idempotent']}")
print(f"  ⚠️  Com erros          : {report['total_errors']}")
print(f"  🔴 Não convertidos    : {report['remaining_unconverted']}")
print(f"  📝 Relatório salvo em : {report_path}")

if remaining_pdfs:
    print(f'\n⚠️  PDFs ainda não convertidos: {len(remaining_pdfs)}')
    for item in remaining_pdfs:
        print(f"   - [{item['folder']}] {item['file']}")
else:
    print('\n🎉 Todos os PDFs das pastas selecionadas foram convertidos!')
    print('   Critério de parada atingido: nenhum PDF restante nas pastas processadas.')